In [16]:
import sys
import pandas as pd
import numpy as np
import re
from pathlib import Path
data_dir = Path("../data/raw/steam_dataset_2025_csv")

applications = pd.read_csv(
    data_dir / "applications.csv",
    low_memory=False
)

applications.shape

games = applications[
    applications["type"].eq("game")
].copy()

games["release_date"] = pd.to_datetime(
    games["release_date"],
    errors="coerce"
)

games["release_year"] = games["release_date"].dt.year

games = games[
    games["release_year"].between(2015, 2024)
].copy()

games.shape

keep_cols = [
    "appid",
    "name",
    "is_free",
    "release_date",
    "required_age",
    "supported_languages",
    "metacritic_score",
    "recommendations_total",
    "mat_supports_windows",
    "mat_supports_mac",
    "mat_supports_linux",
    "mat_initial_price",
    "mat_final_price",
    "mat_discount_percent",
    "mat_currency",
    "mat_achievement_count",
    "created_at",
    "updated_at",
    "release_year",
]

games = games[keep_cols].copy()


def count_languages(x):
    if pd.isna(x):
        return 0

    x = x.split("<br>")[0]
    x = re.sub(r"<[^>]+>", "", x)

    languages = [
        lang.strip()
        for lang in x.split(",")
        if lang.strip()
    ]

    return len(languages)

games["language_count"] = (
    games["supported_languages"]
    .apply(count_languages)
)


games["initial_price_usd"] = np.where(
    games["mat_currency"] == "USD",
    games["mat_initial_price"] / 100,
    np.nan
)

games["final_price_usd"] = np.where(
    games["mat_currency"] == "USD",
    games["mat_final_price"] / 100,
    np.nan
)

app_genres = pd.read_csv(
    data_dir / "application_genres.csv"
)

genres = pd.read_csv(
    data_dir / "genres.csv"
)

genre_map = app_genres.merge(
    genres,
    left_on="genre_id",
    right_on="id",
    how="left"
)

game_genres = genre_map[
    genre_map["appid"].isin(games["appid"])
].copy()

non_game_genres = [
    "Design & Illustration",
    "Utilities",
    "Game Development",
    "Education",
    "Video Production",
    "Software Training",
    "Audio Production",
    "Web Publishing",
    "Animation & Modeling",
]

non_game_appids = game_genres.loc[
    game_genres["name"].isin(non_game_genres),
    "appid"
].unique()

games = games[
    ~games["appid"].isin(non_game_appids)
].copy()


game_genres = genre_map[
    genre_map["appid"].isin(games["appid"])
].copy()

genre_frequency = game_genres[
    "genre_id"
].value_counts()

common_genre_ids = genre_frequency[
    genre_frequency >= 500
].index

game_genres_common = game_genres[
    game_genres["genre_id"].isin(common_genre_ids)
].copy()


genre_features = pd.crosstab(
    game_genres_common["appid"],
    game_genres_common["name"]
)

genre_features = (
    genre_features > 0
).astype(int)

genre_features.columns = [
    "genre_" +
    col.replace(" ", "_").replace("&", "and")
    for col in genre_features.columns
]


games = games.merge(
    genre_features,
    left_on="appid",
    right_index=True,
    how="left"
)

genre_columns = [
    col
    for col in games.columns
    if col.startswith("genre_")
]

games[genre_columns] = (
    games[genre_columns]
    .fillna(0)
    .astype(int)
)


player_history = pd.read_csv(
    "https://raw.githubusercontent.com/"
    "aanomali/steam-multiplayer-forecasting/"
    "main/data/processed/multiplayer_history.csv"
)

player_history["date"] = pd.to_datetime(
    player_history["date"]
)


common_appids = (
    set(games["appid"])
    & set(player_history["appid"])
)

games_with_history = games[
    games["appid"].isin(common_appids)
].copy()

games_with_history.shape


games_with_history["release_month"] = (
    games_with_history["release_date"]
    .dt.to_period("M")
)

games_with_history["month_1"] = (
    games_with_history["release_month"] + 1
)

games_with_history["month_12"] = (
    games_with_history["release_month"] + 12
)

player_history["month"] = (
    player_history["date"]
    .dt.to_period("M")
)


month1 = player_history[
    ["appid", "month", "avg_players", "peak_players"]
].rename(
    columns={
        "month": "month_1",
        "avg_players": "month1_avg_players",
        "peak_players": "month1_peak_players",
    }
)

month12 = player_history[
    ["appid", "month", "avg_players", "peak_players"]
].rename(
    columns={
        "month": "month_12",
        "avg_players": "month12_avg_players",
        "peak_players": "month12_peak_players",
    }
)


model_df = games_with_history.merge(
    month1,
    on=["appid", "month_1"],
    how="left"
)

model_df = model_df.merge(
    month12,
    on=["appid", "month_12"],
    how="left"
)

model_df = model_df.dropna(
    subset=[
        "month1_avg_players",
        "month12_avg_players",
    ]
).copy()

print(model_df.shape)


model_df["retention_ratio"] = (
    model_df["month12_avg_players"]
    / model_df["month1_avg_players"]
)

model_df["log_retention"] = np.log(
    (model_df["month12_avg_players"] + 1)
    /
    (model_df["month1_avg_players"] + 1)
)

processed_dir = Path("../data/processed")
processed_dir.mkdir(
    parents=True,
    exist_ok=True
)

model_df.to_csv(
    processed_dir / "steam_modeling_data.csv",
    index=False
)

print(model_df.shape)

(276, 41)
(276, 43)
